# Imputación de faltantes de DViento (dirección del viento)

Este notebook realiza la imputación de faltantes de **DViento** tratando la variable como **circular**.  
Eso significa que no se modela directamente en grados, sino mediante sus componentes:

- **seno(ángulo)**
- **coseno(ángulo)**

Luego, al final, se reconstruye la dirección en grados.

## ¿Por qué hacer esto?
La dirección del viento tiene un comportamiento angular.  
Por ejemplo:

- 359° y 1° están muy cerca físicamente,
- pero si se modelan como números ordinarios, parecería que están muy lejos.

Por eso, para imputar correctamente la dirección del viento, es mejor trabajar con seno y coseno.


In [1]:

import pandas as pd
import numpy as np
import math
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

VARIABLE = "DViento"
RUTA = "DViento.csv"
ARCHIVO_BASE = "DViento"


In [2]:

df = pd.read_csv(RUTA)
df["Fecha inicial"] = pd.to_datetime(df["Fecha inicial"])
df = df[["Estacion", "Fecha inicial", VARIABLE]].copy()

print("Registros:", len(df))
print("Columnas:", df.columns.tolist())
print("Estaciones:", sorted(df["Estacion"].unique()))
print("Rango temporal:", df["Fecha inicial"].min().date(), "a", df["Fecha inicial"].max().date())
display(df.head())


Registros: 1455
Columnas: ['Estacion', 'Fecha inicial', 'DViento']
Estaciones: ['CENTRO DE ALTO RENDIMIENTO', 'KENNEDY']
Rango temporal: 2020-01-01 a 2021-12-31


,Estacion,Fecha inicial,DViento
0,CENTRO DE ALTO RENDIMIENTO,2021-12-31,258.695652
1,CENTRO DE ALTO RENDIMIENTO,2021-12-30,223.833333
2,CENTRO DE ALTO RENDIMIENTO,2021-12-29,274.416667
3,CENTRO DE ALTO RENDIMIENTO,2021-12-28,191.041667
4,CENTRO DE ALTO RENDIMIENTO,2021-12-27,188.208333


In [3]:

def sin_cos_to_angle(s, c):
    ang = np.rad2deg(np.arctan2(s, c))
    return (ang + 360) % 360

def build_full(df_in, var):
    estaciones = sorted(df_in["Estacion"].unique())
    fechas = pd.date_range(df_in["Fecha inicial"].min(), df_in["Fecha inicial"].max(), freq="D")
    grid = pd.MultiIndex.from_product([estaciones, fechas], names=["Estacion", "Fecha inicial"]).to_frame(index=False)

    out = grid.merge(df_in[["Estacion", "Fecha inicial", var]], on=["Estacion", "Fecha inicial"], how="left")
    out["original_missing"] = out[var].isna()

    out["target_sin"] = np.where(out[var].notna(), np.sin(np.deg2rad(out[var])), np.nan)
    out["target_cos"] = np.where(out[var].notna(), np.cos(np.deg2rad(out[var])), np.nan)
    return out

def add_features(base, var):
    df_feat = base.copy().sort_values(["Estacion", "Fecha inicial"]).reset_index(drop=True)

    fecha = df_feat["Fecha inicial"]
    df_feat["year"] = fecha.dt.year
    df_feat["month"] = fecha.dt.month
    df_feat["day"] = fecha.dt.day
    df_feat["dayofweek"] = fecha.dt.dayofweek
    df_feat["weekofyear"] = fecha.dt.isocalendar().week.astype(int)
    df_feat["dayofyear"] = fecha.dt.dayofyear

    # codificación cíclica del tiempo
    df_feat["sin_doy"] = np.sin(2 * np.pi * df_feat["dayofyear"] / 365.25)
    df_feat["cos_doy"] = np.cos(2 * np.pi * df_feat["dayofyear"] / 365.25)
    df_feat["sin_dow"] = np.sin(2 * np.pi * df_feat["dayofweek"] / 7)
    df_feat["cos_dow"] = np.cos(2 * np.pi * df_feat["dayofweek"] / 7)
    df_feat["sin_month"] = np.sin(2 * np.pi * df_feat["month"] / 12)
    df_feat["cos_month"] = np.cos(2 * np.pi * df_feat["month"] / 12)

    # información de otras estaciones el mismo día
    resumen_fecha = df_feat.groupby("Fecha inicial")[["target_sin", "target_cos"]].agg(["sum", "count"])
    resumen_fecha.columns = ["sum_sin", "count_sin", "sum_cos", "count_cos"]
    df_feat = df_feat.merge(resumen_fecha, left_on="Fecha inicial", right_index=True, how="left")

    self_sin_obs = df_feat["target_sin"].notna().astype(int)
    self_cos_obs = df_feat["target_cos"].notna().astype(int)

    sin_val = df_feat["target_sin"].fillna(0)
    cos_val = df_feat["target_cos"].fillna(0)

    denom_s = df_feat["count_sin"] - self_sin_obs
    denom_c = df_feat["count_cos"] - self_cos_obs

    df_feat["other_station_sin"] = np.where(denom_s > 0, (df_feat["sum_sin"] - sin_val) / denom_s, np.nan)
    df_feat["other_station_cos"] = np.where(denom_c > 0, (df_feat["sum_cos"] - cos_val) / denom_c, np.nan)

    df_feat.drop(columns=["sum_sin", "count_sin", "sum_cos", "count_cos"], inplace=True)

    # rezagos y medias móviles
    lags = [1, 2, 3, 7, 14]
    ventanas = [3, 7, 14]

    for estacion, sub in df_feat.groupby("Estacion"):
        sub = sub.sort_values("Fecha inicial")

        for lag in lags:
            df_feat.loc[sub.index, f"lag_{lag}_sin"] = sub["target_sin"].shift(lag).values
            df_feat.loc[sub.index, f"lag_{lag}_cos"] = sub["target_cos"].shift(lag).values

        for w in ventanas:
            df_feat.loc[sub.index, f"rollmean_{w}_sin"] = sub["target_sin"].shift(1).rolling(w, min_periods=1).mean().values
            df_feat.loc[sub.index, f"rollmean_{w}_cos"] = sub["target_cos"].shift(1).rolling(w, min_periods=1).mean().values

    return df_feat

def get_feature_cols(df_feat, var):
    return [c for c in df_feat.columns if c not in ["Estacion", "Fecha inicial", var, "original_missing", "target_sin", "target_cos"]]

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestRegressor(
            n_estimators=120,
            max_depth=10,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

def impute_circular_rf(df_in, var):
    base = build_full(df_in, var)
    feat = add_features(base, var)
    feature_cols = get_feature_cols(feat, var)

    train = feat[feat[var].notna()].copy()

    model_sin = make_model()
    model_cos = make_model()

    model_sin.fit(train[feature_cols], train["target_sin"])
    model_cos.fit(train[feature_cols], train["target_cos"])

    work = base.copy()
    missing_idx = work[work[var].isna()].sort_values(["Estacion", "Fecha inicial"]).index.tolist()

    for idx in missing_idx:
        wf = add_features(work, var)
        X = wf.loc[[idx], feature_cols]

        pred_sin = float(model_sin.predict(X)[0])
        pred_cos = float(model_cos.predict(X)[0])

        norma = math.sqrt(pred_sin**2 + pred_cos**2)
        if norma == 0 or np.isnan(norma):
            pred_sin, pred_cos, norma = 0.0, 1.0, 1.0

        pred_sin /= norma
        pred_cos /= norma

        angulo = float(sin_cos_to_angle(pred_sin, pred_cos))

        work.loc[idx, var] = angulo
        work.loc[idx, "target_sin"] = pred_sin
        work.loc[idx, "target_cos"] = pred_cos

    return work

def angular_errors(y_true, y_pred):
    diff = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    return np.where(diff > 180, 360 - diff, diff)


In [4]:

# Diagnóstico inicial de faltantes
fechas_completas = pd.date_range(df["Fecha inicial"].min(), df["Fecha inicial"].max(), freq="D")

resumen_faltantes = []
for estacion, sub in df.groupby("Estacion"):
    faltantes = sorted(set(fechas_completas) - set(sub["Fecha inicial"]))
    resumen_faltantes.append({
        "Estacion": estacion,
        "Registros_observados": len(sub),
        "Dias_esperados": len(fechas_completas),
        "Dias_faltantes": len(faltantes),
        "Porcentaje_faltante": round(len(faltantes) / len(fechas_completas) * 100, 4)
    })

resumen_faltantes = pd.DataFrame(resumen_faltantes)
display(resumen_faltantes)


,Estacion,Registros_observados,Dias_esperados,Dias_faltantes,Porcentaje_faltante
0,CENTRO DE ALTO RENDIMIENTO,731,731,0,0.0000
1,KENNEDY,724,731,7,0.9576


In [5]:

# Validación por ocultamiento de bloques reales observados en KENNEDY
ken_full = build_full(df, VARIABLE)
ken_full = ken_full[ken_full["Estacion"] == "KENNEDY"].sort_values("Fecha inicial").reset_index(drop=True)

bloques_validacion = [(3, 40), (3, 180), (3, 320), (4, 430), (4, 560), (4, 650)]

resultados_validacion = []

for tam_bloque, inicio in bloques_validacion:
    fechas_bloque = ken_full.loc[inicio:inicio + tam_bloque - 1, "Fecha inicial"]
    valores_bloque = ken_full.loc[inicio:inicio + tam_bloque - 1, VARIABLE]

    if valores_bloque.isna().any():
        continue

    tmp = df.copy()
    mask = (tmp["Estacion"] == "KENNEDY") & (tmp["Fecha inicial"].isin(fechas_bloque))

    verdaderos = tmp.loc[mask, VARIABLE].tolist()
    tmp = tmp.loc[~mask].copy()

    imputado_tmp = impute_circular_rf(tmp, VARIABLE)
    predichos = (
        imputado_tmp[
            (imputado_tmp["Estacion"] == "KENNEDY") &
            (imputado_tmp["Fecha inicial"].isin(fechas_bloque))
        ]
        .sort_values("Fecha inicial")[VARIABLE]
        .tolist()
    )

    errores = angular_errors(verdaderos, predichos)

    resultados_validacion.append({
        "inicio": str(fechas_bloque.min().date()),
        "fin": str(fechas_bloque.max().date()),
        "tam_bloque": tam_bloque,
        "MAE_angular": float(np.mean(errores)),
        "RMSE_angular": float(np.sqrt(np.mean(np.square(errores))))
    })

tabla_validacion = pd.DataFrame(resultados_validacion)
display(tabla_validacion)

print("MAE angular promedio:", round(tabla_validacion["MAE_angular"].mean(), 4))
print("RMSE angular promedio:", round(tabla_validacion["RMSE_angular"].mean(), 4))


,inicio,fin,tam_bloque,MAE_angular,RMSE_angular
0,2020-02-10,2020-02-12,3,53.188079,60.747187
1,2020-06-29,2020-07-01,3,55.333714,57.041948
2,2020-11-16,2020-11-18,3,34.068104,42.494787
3,2021-03-06,2021-03-09,4,28.081346,28.886269
4,2021-07-14,2021-07-17,4,13.437389,15.463694
5,2021-10-12,2021-10-15,4,27.176866,29.998723


MAE angular promedio: 35.2142
RMSE angular promedio: 39.1054


In [6]:

# Imputación final de los faltantes reales
resultado_imputado = impute_circular_rf(df, VARIABLE)

tabla_imputados = resultado_imputado.loc[
    resultado_imputado["original_missing"],
    ["Estacion", "Fecha inicial", VARIABLE]
].copy()

display(tabla_imputados)


,Estacion,Fecha inicial,DViento
1231,KENNEDY,2021-05-15,152.720544
1232,KENNEDY,2021-05-16,170.699837
1233,KENNEDY,2021-05-17,194.933295
1275,KENNEDY,2021-06-28,192.265060
1276,KENNEDY,2021-06-29,144.589686
1277,KENNEDY,2021-06-30,170.554279
1278,KENNEDY,2021-07-01,216.236334


In [7]:

# Construcción de datasets finales para exportación
resultado_base = build_full(df, VARIABLE)

resultado_final = resultado_base.merge(
    resultado_imputado[["Estacion", "Fecha inicial", VARIABLE]].rename(columns={VARIABLE: f"{VARIABLE}_final"}),
    on=["Estacion", "Fecha inicial"],
    how="left"
)

resultado_final[f"{VARIABLE}_original"] = resultado_final[VARIABLE]
resultado_final[f"{VARIABLE}_imputado"] = np.where(resultado_final["original_missing"], resultado_final[f"{VARIABLE}_final"], np.nan)
resultado_final["Fue_faltante"] = resultado_final["original_missing"]
resultado_final["Fuente_valor"] = np.where(resultado_final["Fue_faltante"], "Imputado", "Original")
resultado_final["Fecha final"] = ""

resultado_exportar_detallado = resultado_final[
    [
        "Estacion",
        "Fecha inicial",
        "Fecha final",
        f"{VARIABLE}_original",
        f"{VARIABLE}_imputado",
        f"{VARIABLE}_final",
        "Fue_faltante",
        "Fuente_valor",
    ]
].copy()

resultado_exportar_detallado = (
    resultado_exportar_detallado
    .sort_values(["Estacion", "Fecha inicial"])
    .reset_index(drop=True)
)

resultado_exportar_final = resultado_exportar_detallado[
    ["Estacion", "Fecha inicial", "Fecha final", f"{VARIABLE}_final", "Fue_faltante", "Fuente_valor"]
].rename(columns={f"{VARIABLE}_final": VARIABLE})

tabla_imputados_exportar = resultado_exportar_detallado.loc[
    resultado_exportar_detallado["Fue_faltante"],
    [
        "Estacion",
        "Fecha inicial",
        "Fecha final",
        f"{VARIABLE}_original",
        f"{VARIABLE}_imputado",
        f"{VARIABLE}_final",
        "Fuente_valor",
    ],
].copy()

display(resultado_exportar_detallado.head())
display(tabla_imputados_exportar)


,Estacion,Fecha inicial,Fecha final,DViento_original,DViento_imputado,DViento_final,Fue_faltante,Fuente_valor
0,CENTRO DE ALTO RENDIMIENTO,2020-01-01,,251.333333,NaN,251.333333,False,Original
1,CENTRO DE ALTO RENDIMIENTO,2020-01-02,,273.625000,NaN,273.625000,False,Original
2,CENTRO DE ALTO RENDIMIENTO,2020-01-03,,255.875000,NaN,255.875000,False,Original
3,CENTRO DE ALTO RENDIMIENTO,2020-01-04,,298.958333,NaN,298.958333,False,Original
4,CENTRO DE ALTO RENDIMIENTO,2020-01-05,,260.500000,NaN,260.500000,False,Original


,Estacion,Fecha inicial,Fecha final,DViento_original,DViento_imputado,DViento_final,Fuente_valor
1231,KENNEDY,2021-05-15,,NaN,152.720544,152.720544,Imputado
1232,KENNEDY,2021-05-16,,NaN,170.699837,170.699837,Imputado
1233,KENNEDY,2021-05-17,,NaN,194.933295,194.933295,Imputado
1275,KENNEDY,2021-06-28,,NaN,192.265060,192.265060,Imputado
1276,KENNEDY,2021-06-29,,NaN,144.589686,144.589686,Imputado
1277,KENNEDY,2021-06-30,,NaN,170.554279,170.554279,Imputado
1278,KENNEDY,2021-07-01,,NaN,216.236334,216.236334,Imputado


In [8]:

# Exportación de archivos
resultado_exportar_detallado.to_csv(f"{ARCHIVO_BASE}_dataset_completo_imputado_detallado.csv", index=False)
resultado_exportar_final.to_csv(f"{ARCHIVO_BASE}_dataset_completo_final.csv", index=False)
tabla_imputados_exportar.to_csv(f"{ARCHIVO_BASE}_valores_imputados.csv", index=False)

with pd.ExcelWriter(f"{ARCHIVO_BASE}_dataset_completo_imputado_detallado.xlsx", engine="openpyxl") as writer:
    resultado_exportar_detallado.to_excel(writer, sheet_name="detalle_completo", index=False)
    resultado_exportar_final.to_excel(writer, sheet_name="dataset_final", index=False)
    tabla_imputados_exportar.to_excel(writer, sheet_name="solo_imputados", index=False)

print("Archivos exportados correctamente:")
print(f"- {ARCHIVO_BASE}_dataset_completo_imputado_detallado.csv")
print(f"- {ARCHIVO_BASE}_dataset_completo_final.csv")
print(f"- {ARCHIVO_BASE}_valores_imputados.csv")
print(f"- {ARCHIVO_BASE}_dataset_completo_imputado_detallado.xlsx")


Archivos exportados correctamente:
- DViento_dataset_completo_imputado_detallado.csv
- DViento_dataset_completo_final.csv
- DViento_valores_imputados.csv
- DViento_dataset_completo_imputado_detallado.xlsx


## Interpretación metodológica final

- La imputación se realizó con **Random Forest**.
- Como `DViento` es una variable angular, se modeló usando **seno** y **coseno**.
- Después se reconstruyó el ángulo final en grados.
- Esto evita errores conceptuales que aparecen cuando se trata la dirección como una variable lineal común.

### Archivos exportados
1. **dataset detallado**: original, imputado y final
2. **dataset final**: listo para usar en análisis e integración
3. **solo imputados**: únicamente las filas que fueron completadas
4. **archivo Excel** con las tres hojas
